This notebook loads and preprocesses the Titanic dataset, trains Random Forest models, evaluates feature importance, tunes `n_estimators` and `max_depth`, and reports the best-performing model.

In [2]:
# importing necessary libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load data
df = pd.read_csv("titanic.csv")

# Separate target and features
y = df["Survived"]
X = df.drop(columns=["Survived"])

# Drop non important columns
X = X.drop(columns=["Name", "Ticket", "Cabin"])

# Identify column types
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


C:\Users\ndabambil\AppData\Local\Temp\ipykernel_18080\4205640147.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()


In [3]:

# Baseline Random Forest
baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("rf", RandomForestClassifier(random_state=42))
])

baseline_model.fit(X_train, y_train)
baseline_preds = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(y_test, baseline_preds)
print(f"Baseline Accuracy: {baseline_accuracy:.4f}")


Baseline Accuracy: 0.7989


In [4]:

# Feature Importance

rf_model = baseline_model.named_steps["rf"]

feature_names = baseline_model.named_steps["preprocessor"].get_feature_names_out()
importances = rf_model.feature_importances_

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values("Importance", ascending=False)

importance_df.head(15)


,Feature,Importance
0,num__PassengerId,0.190002
5,num__Fare,0.189782
2,num__Age,0.159172
7,cat__Sex_male,0.148028
6,cat__Sex_female,0.134419
1,num__Pclass,0.075410
3,num__SibSp,0.037030
4,num__Parch,0.029598
10,cat__Embarked_S,0.018091
8,cat__Embarked_C,0.011597


In [5]:

# Most important feature
top_feature = importance_df.iloc[0]
print("Most Important Feature:")
print(top_feature)


Most Important Feature:
Feature       num__PassengerId
Importance            0.190002
Name: 0, dtype: object


In [6]:

# Hyperparameter tuning

results = []

for n_estimators in [50, 100, 200, 300]:
    for max_depth in [None, 3, 5, 10, 15]:

        model = Pipeline([
            ("preprocessor", preprocessor),
            ("rf", RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                random_state=42
            ))
        ])

        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        acc = accuracy_score(y_test, preds)

        results.append({
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "accuracy": acc
        })

results_df = pd.DataFrame(results).sort_values(
    "accuracy", ascending=False
)

results_df


,n_estimators,max_depth,accuracy
8,100,10.0,0.815642
13,200,10.0,0.810056
9,100,15.0,0.810056
18,300,10.0,0.810056
19,300,15.0,0.804469
16,300,3.0,0.804469
2,50,5.0,0.804469
1,50,3.0,0.804469
11,200,3.0,0.804469
15,300,NaN,0.804469


In [7]:

# Best model

best_model_result = results_df.iloc[0]

print("Best Model")
print("----------")
print(f"Accuracy     : {best_model_result['accuracy']:.4f}")
print(f"n_estimators : {best_model_result['n_estimators']}")
print(f"max_depth    : {best_model_result['max_depth']}")


Best Model
----------
Accuracy     : 0.8156
n_estimators : 100.0
max_depth    : 10.0
